# 📸 템플릿 1 — 사진 → 캐릭터화

업로드한 사진을 디즈니/지브리/유화 등 다양한 스타일로 변환하는 앱.

## 사용 모델
**Stable Diffusion XL Turbo** (4 step만에 이미지 생성, Colab T4에서 동작)

## 학생이 가장 쉽게 손댈 곳
- `STYLES` 딕셔너리: 새 스타일 추가 / 프롬프트 튜닝
- `strength` 슬라이더 범위 조정
- UI 디자인 변경 (`gr.Blocks`)

⚠️ **실행 전**: 메뉴 → 런타임 → 런타임 유형 변경 → **GPU (T4)** 선택

In [ ]:
# 필수 라이브러리 설치 (1-2분)
!pip install -q diffusers transformers accelerate gradio

In [ ]:
import torch
from diffusers import AutoPipelineForImage2Image
import gradio as gr
from PIL import Image

# 모델 로드 (처음엔 ~14GB 다운로드, 5-10분 소요)
print("모델 로딩 중...")
pipe = AutoPipelineForImage2Image.from_pretrained(
    "stabilityai/sdxl-turbo",
    torch_dtype=torch.float16,
    variant="fp16",
)
pipe.enable_model_cpu_offload()   # T4 16GB에 맞게 메모리 절약
print("✅ 모델 로딩 완료")

In [ ]:
# ===== 학생이 마음대로 추가/수정할 부분 =====

STYLES = {
    "🎨 디즈니 픽사": "disney pixar 3d animation style, vibrant colors, big expressive eyes, detailed",
    "🌸 지브리 애니메": "studio ghibli anime style, soft watercolor, dreamy atmosphere, pastel colors",
    "🖼️ 고전 유화": "classical oil painting, rembrandt lighting, detailed brushwork, baroque",
    "📺 한국 웹툰": "korean manhwa webtoon style, vibrant, sharp lines, anime aesthetic",
    "🌃 사이버펑크": "cyberpunk style, neon lights, futuristic, vibrant magenta and cyan",
    "🎭 수채화": "watercolor portrait, soft pastel, dreamy, artistic, paper texture",
    # ← 여기에 새 스타일 자유롭게 추가하기
    # "🍂 가을느낌": "autumn aesthetic, warm tones, vintage film, melancholic",
}

In [ ]:
def transform(image, style, strength):
    """이미지를 선택된 스타일로 변환"""
    if image is None:
        return None
    
    # 메모리 안전을 위해 512x512로 리사이즈
    image = image.convert("RGB")
    image.thumbnail((512, 512), Image.LANCZOS)
    
    prompt = f"portrait photo of a person, {STYLES[style]}, masterpiece, best quality"
    
    result = pipe(
        prompt=prompt,
        image=image,
        strength=strength,            # 낮으면 원본과 비슷, 높으면 스타일 강함
        guidance_scale=0.0,           # SDXL Turbo는 0이 정석
        num_inference_steps=4,        # Turbo는 4 step으로 충분
    ).images[0]
    
    return result

# Gradio UI
with gr.Blocks(title="📸 AI 캐릭터화", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 📸 사진 → 캐릭터화\n사진을 업로드하고 스타일을 골라보세요")
    with gr.Row():
        with gr.Column():
            img_in = gr.Image(type="pil", label="원본 사진", height=400)
            style = gr.Dropdown(list(STYLES.keys()), value=list(STYLES.keys())[0], label="🎨 스타일")
            strength = gr.Slider(0.3, 0.8, 0.6, step=0.05, label="✨ 변형 강도 (낮을수록 원본 유지)")
            btn = gr.Button("✨ 변환하기", variant="primary", size="lg")
        with gr.Column():
            img_out = gr.Image(label="결과", height=400)
    
    btn.click(transform, [img_in, style, strength], img_out)

demo.launch(share=True, debug=False)
# 콘솔에 나오는 https://*.gradio.live URL을 친구한테 공유하세요!

---
## 🚀 바이브 코딩 확장 아이디어

### 쉬움 (Gemini에게 부탁)
- 스타일 추가 ("게임 픽셀아트", "수묵화", "캐리커처")
- 배치 처리: 한 사진을 여러 스타일로 동시 변환
- 결과를 ZIP으로 다운로드 버튼

### 중간
- "원본 vs 결과" 슬라이더 비교 뷰
- 같은 사진 + 같은 스타일로 다양한 시드 생성
- 텍스트 프롬프트 추가 입력 ("선글라스 끼고", "겨울 배경" 등)

### 도전적
- 얼굴 보존 강화: IP-Adapter나 InstantID 적용
- 동영상 입력 → 프레임별 변환
- 영어 프롬프트만 받는 SDXL에 한국어 입력 받아서 자동 번역

### 🎁 자랑하기 팁
- 본인 인스타 프로필을 다양한 스타일로 만들어서 시리즈로 업로드
- 친구들 사진을 디즈니화해서 단톡방에 뿌리기